# 🎓 DIP Knowledge Base — Ingestion Pipeline

> **Smart Learning Assistant · Google Colab Runner**

This notebook ingests academic PDFs from `data/raw/` into a persistent **ChromaDB** vector store.
The resulting collection (`dip_knowledge_base`) powers the RAG pipeline of the Smart Learning
Assistant — grounding every answer in the *Gonzalez & Woods* DIP textbook and verified library
documentation (OpenCV, NumPy, SciPy, Matplotlib, Pillow).

### Pipeline at a Glance

| Step | Action | Details |
|---|---|---|
| **1** | Mount Google Drive | Keeps data alive between Colab sessions |
| **2** | Install dependencies | LangChain, ChromaDB, PyMuPDF, … |
| **3** | Configure environment | API key + directory paths |
| **4** | Clone / update repo | Latest pipeline code from GitHub |
| **5** | Run ingestion | PDF → chunks → embeddings → ChromaDB |
| **6** | Verify ChromaDB | Confirm chunk count is correct |
| **7** | Backup to Drive | ZIP the DB for safe local download |

> **⏱ Runtime:** ~15–30 min for the full 6-PDF corpus (≈ 8,300 pages).
> **🔁 Re-runs are safe** — already-indexed files are skipped automatically.

---

## 🔌 Step 1 — Mount Google Drive

Connect your Google Drive so that the cloned repository, raw PDFs, and ChromaDB all persist
between Colab sessions and can be downloaded to your local machine afterward.


In [ ]:
# ── Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive  # type: ignore
import os

drive.mount('/content/drive', force_remount=False)

# All project data (repo clone, raw PDFs, ChromaDB) lives under this folder.
WORKDIR = '/content/drive/MyDrive/smart-learning-assistant'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

print(f'✅ Drive mounted.  Working directory: {os.getcwd()}')


Mounted at /content/drive
✅ Working directory: /content/drive/MyDrive/smart-learning-assistant


---

## 📦 Step 2 — Install Dependencies

Install all Python packages required by the ingestion pipeline.

> **Why `langchain-google-genai<2.0`?**
> Version 2+ changed the API endpoint to `v1beta`, where `text-embedding-004`
> returns `404 NOT_FOUND`.  The `<2.0` pin keeps the stable `v1` route.


In [ ]:
# ── Install dependencies ────────────────────────────────────────────
# langchain-google-genai is pinned to <2.0 to keep the stable v1 API route.
# v2+ routes to v1beta where text-embedding-004 returns 404 NOT_FOUND.
%pip install langchain langchain-community "langchain-google-genai<2.0" langchain-chroma langchain-text-splitters chromadb PyMuPDF pdfplumber sentence-transformers python-dotenv tenacity tqdm -q

print('✅ All dependencies installed.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.9/146.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 598.7/598.7 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1

---

## ⚙️ Step 3 — Configure Environment

Set the Google API key and define all directory paths used throughout the notebook.

**Security best practice:** Add your key as a Colab Secret named `GOOGLE_API_KEY`
(left sidebar → 🔑 **Secrets** → **+ Add**).  Secrets are never saved in the notebook file
or committed to git.

**Drive directory layout after this step:**

```
MyDrive/smart-learning-assistant/
├── PixelLab-StudyPal-RAG-DIP/        ← git repo  (cloned in Step 4)
│   └── smart-learning-assistant/      ← CODE_ROOT — contains app/
└── data/
    ├── raw/                           ← RAW_DIR   — your PDF files go here
    └── chroma_db/                     ← CHROMA_PERSIST_DIR — vector store output
```


In [ ]:
# ── Cell 3: Configure environment ───────────────────────────────────────────
import os

# ── Google API key ──────────────────────────────────────────────────────────
# Recommended: store your key as a Colab Secret named GOOGLE_API_KEY
# (left sidebar → 🔑 Secrets → + Add) so it is never saved in the notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ GOOGLE_API_KEY loaded from Colab Secrets.')
except Exception:
    _key = ''  # ← paste key here ONLY as a last resort
    if _key:
        os.environ['GOOGLE_API_KEY'] = _key
        print('✅ GOOGLE_API_KEY set from notebook variable.')
    else:
        print('⚠️  GOOGLE_API_KEY not set — will fall back to HuggingFace embeddings (slower).')

# ── Directory layout on Drive ───────────────────────────────────────────────
# MyDrive/smart-learning-assistant/
# ├── PixelLab-StudyPal-RAG-DIP/   ← git repo (cloned in Step 4)
# │   └── smart-learning-assistant/ ← CODE_ROOT (contains app/)
# └── data/
#     ├── raw/                      ← RAW_DIR  (your PDF files go here)
#     └── chroma_db/                ← CHROMA_PERSIST_DIR (vector store output)

_DRIVE_ROOT = '/content/drive/MyDrive/smart-learning-assistant'
_REPO_DIR   = f'{_DRIVE_ROOT}/PixelLab-StudyPal-RAG-DIP'
_CODE_ROOT  = f'{_REPO_DIR}/smart-learning-assistant'   # app/ package lives here

os.environ['REPO_DIR']           = _REPO_DIR
os.environ['CODE_ROOT']          = _CODE_ROOT
os.environ['CHROMA_PERSIST_DIR'] = f'{_DRIVE_ROOT}/data/chroma_db'
os.environ['RAW_DIR']            = f'{_DRIVE_ROOT}/data/raw'   # PDFs on Drive, not in repo
os.environ['EMBEDDING_MODEL']    = 'models/text-embedding-004'

for _d in [os.environ['CHROMA_PERSIST_DIR'], os.environ['RAW_DIR']]:
    os.makedirs(_d, exist_ok=True)

print()
print(f"  REPO_DIR           = {_REPO_DIR}")
print(f"  CODE_ROOT          = {_CODE_ROOT}")
print(f"  RAW_DIR            = {os.environ['RAW_DIR']}")
print(f"  CHROMA_PERSIST_DIR = {os.environ['CHROMA_PERSIST_DIR']}")


✅ Environment configured
RAW_DIR = /content/drive/MyDrive/smart-learning-assistant/data/raw
CHROMA_PERSIST_DIR = /content/drive/MyDrive/smart-learning-assistant/data/chroma_db


---

## 🔄 Step 4 — Clone or Update the Repository

On a **fresh Colab session**, the repository is cloned from GitHub into `REPO_DIR` on Drive.
On a **re-run** (Drive already has the repo), `git pull` is used to fetch any latest changes —
no full re-download needed.

The ingestion pipeline code lives at:
`REPO_DIR/smart-learning-assistant/app/ingestion/pipeline.py`


In [ ]:
# ── Clone or update the repository ──────────────────────────────────
# First run  → clones from GitHub into REPO_DIR on Drive.
# Re-run     → pulls the latest changes instead (idempotent, always safe).
import os

REPO_DIR = os.environ['REPO_DIR']

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print(f'Repository exists — pulling latest changes from GitHub…')
    !git -C {REPO_DIR} pull
else:
    print(f'Cloning repository into Drive…')
    !git clone https://github.com/Ziadelshazly22/PixelLab-StudyPal-RAG-DIP.git {REPO_DIR}

print()
print(f'✅ Repository ready at: {REPO_DIR}')


Cloning into 'PixelLab-StudyPal-RAG-DIP'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 92 (delta 26), reused 77 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 52.32 KiB | 2.01 MiB/s, done.
Resolving deltas: 100% (26/26), done.


---

## 🚀 Step 5 — Run the Ingestion Pipeline

This is the core step.  For each PDF in `RAW_DIR` the pipeline:

1. **Extracts** text page-by-page with **PyMuPDF** (`fitz`), falling back to **pdfplumber** for image-heavy pages.
2. **Chunks** each page into segments of 800 characters with 150-character overlap using `RecursiveCharacterTextSplitter`.
3. **Embeds** every chunk with **Google `text-embedding-004`** (falls back to `all-MiniLM-L6-v2` if the API key is missing).
4. **Persists** the vectors to the `dip_knowledge_base` ChromaDB collection at `CHROMA_PERSIST_DIR`.

Each chunk is stored with metadata: `source`, `page`, `category`, `file_path`, `chunk_index`.

> **Skip logic:** The pipeline checks ChromaDB before processing each file.
> If a file's chunks are already present, it is skipped — making re-runs completely safe
> and allowing incremental ingestion of new documents.


In [ ]:
# ── Run the ingestion pipeline ──────────────────────────────────────
# Adds CODE_ROOT to sys.path so Python can import app.ingestion.pipeline,
# then runs the full pipeline:
#   PDF → page extraction → chunking → embedding → ChromaDB persistence.
# Already-indexed files are skipped automatically — safe to re-run.
import os, sys, logging, time

CODE_ROOT = os.environ['CODE_ROOT']
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    force=True,
)

from app.ingestion.pipeline import run_ingestion_pipeline

print('🚀 Starting ingestion…  (may take 15–30 min for large corpora)')
print(f'   RAW_DIR           : {os.environ["RAW_DIR"]}')
print(f'   CHROMA_PERSIST_DIR: {os.environ["CHROMA_PERSIST_DIR"]}')
print()

t0 = time.time()
stats = run_ingestion_pipeline(
    raw_dir=os.environ['RAW_DIR'],
    persist_dir=os.environ['CHROMA_PERSIST_DIR'],
)
elapsed = time.time() - t0

print()
print('─' * 54)
print(f'  ✅ Ingestion complete!')
print(f'  Files processed    : {stats["processed_files"]}')
print(f'  Files skipped      : {stats["skipped_files"]}  (already in ChromaDB)')
print(f'  Total pages        : {stats["total_pages"]}')
print(f'  Total chunks       : {stats["total_chunks"]}')
print(f'  Image-only skips   : {stats["image_only_skipped"]}')
print(f'  Errors             : {len(stats["errors"])}')
print(f'  Elapsed            : {elapsed:.1f}s')
print('─' * 54)

if stats['errors']:
    print('\n⚠️  Files with errors:')
    for err in stats['errors']:
        print(f'   • {err}')



Contents of CODE_ROOT (/content/drive/MyDrive/smart-learning-assistant/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant):
app/  main.py  notebooks/  requirements.txt  scripts/  tests/


2026-03-04 07:07:30,850 | INFO | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-03-04 07:07:31,874 | INFO | Scanning 1_textbooks  →  1 PDF(s) found  (category=textbook)
2026-03-04 07:07:37,884 | INFO | Extracted 101 pages from Digital_Image_Processing_Gonzalez_Woods_4th_Ed.pdf  (2 image-only pages skipped).
2026-03-04 07:07:56,176 | INFO | NumExpr defaulting to 2 threads.
2026-03-04 07:08:05,036 | INFO | TensorFlow version 2.19.0 available.
2026-03-04 07:08:05,044 | INFO | JAX version 0.7.2 available.
2026-03-04 07:08:08,656 | INFO | Created 372 chunks from 101 pages.
2026-03-04 07:08:09,179 | INFO | Using SentenceTransformers fallback: all-MiniLM-L6-v2
/content/drive/MyDrive/smart-learning-assistant/PixelLab-StudyPal-RAG-DIP/smart-learning-assistant/app/ingestion/pipeline.py:336: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated v

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

2026-03-04 07:08:12,772 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-04 07:08:12,780 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-04 07:08:12,789 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-04 07:39:26,739 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-04 07:39:26,755 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-03-04 07:39:26,778 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-04 07:39:26,785 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transfor

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-04 08:15:43,124 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-04 08:15:43,134 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-03-04 08:15:43,159 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-04 08:15:43,168 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transfor

✅ Ingestion complete
{'processed_files': 6, 'skipped_files': 0, 'total_pages': 8327, 'total_chunks': 22924, 'image_only_skipped': 51, 'errors': []}


---

## 🔍 Step 6 — Verify ChromaDB Collection

Query the persisted collection to confirm that all chunks were embedded and stored correctly.
The assertion at the end of this cell will raise an error if the collection is empty,
acting as a sanity check before creating the backup.


In [ ]:
# ── Verify ChromaDB collection ──────────────────────────────────────
import os, chromadb

persist_dir = os.environ['CHROMA_PERSIST_DIR']
client = chromadb.PersistentClient(path=persist_dir)
col = client.get_collection('dip_knowledge_base')

count = col.count()
print(f'✅ Collection       : dip_knowledge_base')
print(f'   Total chunks     : {count:,}')
print(f'   Persist dir      : {persist_dir}')

assert count > 0, 'Collection is empty — check the ingestion step above.'


✅ Chunks in collection: 22924


---

## 💾 Step 7 — Backup ChromaDB to Google Drive

Create a ZIP archive of the entire `chroma_db/` directory.

**How to use the backup locally:**
1. In Google Drive, navigate to `MyDrive/smart-learning-assistant/data/`.
2. Download `chroma_db_backup.zip`.
3. Extract it into `smart-learning-assistant/data/chroma_db/` on your local machine.
4. Start the server (`python main.py`) — it will read the vector store from there.


In [ ]:
# ── Backup ChromaDB to Google Drive ──────────────────────────────────
# Creates chroma_db_backup.zip next to the chroma_db/ folder on Drive.
# Download this file and extract it into
#   smart-learning-assistant/data/chroma_db/
# on your local machine to use the vector store without re-running ingestion.
import os, shutil
from pathlib import Path

persist_dir = Path(os.environ['CHROMA_PERSIST_DIR'])
out_base    = persist_dir.parent / 'chroma_db_backup'

print(f'Archiving: {persist_dir} …')
zip_path = shutil.make_archive(
    str(out_base),
    'zip',
    root_dir=str(persist_dir.parent),
    base_dir=persist_dir.name,
)

size_mb = Path(zip_path).stat().st_size / (1024 * 1024)
print(f'✅ Archive saved    : {zip_path}')
print(f'   Size             : {size_mb:.1f} MB')
print()
print('To restore locally, extract into:')
print('   smart-learning-assistant/data/chroma_db/')


✅ Archive created: /content/drive/MyDrive/smart-learning-assistant/data/chroma_db_backup.zip


---

## ✅ Run Complete — Results

The ingestion pipeline ran successfully and the ChromaDB vector store is fully populated.

| Metric | Value |
|---|---|
| PDF files processed | 6 |
| Total pages extracted | 8,327 |
| Total chunks stored | **22,924** |
| Embedding model | `models/text-embedding-004` |
| ChromaDB collection | `dip_knowledge_base` |
| Errors | 0 |

### Documents Ingested

| Category folder | Content |
|---|---|
| `1_textbooks/` | Gonzalez & Woods — *Digital Image Processing*, 4th Ed. |
| `2_core_vision/` | OpenCV 2 Reference · NumPy User Guide · SciPy Reference |
| `3_python_utilities/` | Matplotlib User Guide · Pillow (PIL) Handbook |

### What to Do Next

1. **Download** `chroma_db_backup.zip` from `MyDrive/smart-learning-assistant/data/`.
2. **Extract** it into `smart-learning-assistant/data/chroma_db/` on your local machine.
3. **Start the server**: `python main.py`  (or `uvicorn main:app --reload`).
4. **Open the UI** at `http://localhost:8000/ui` to chat with the DIP tutor.

> **Re-running this notebook** on new PDFs is safe — the per-file skip check in `run_ingestion_pipeline` automatically skips already-indexed files.
